# Монтрирование Google Drive (нужно для персистентного хранения данных - например, датасетов)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install scapy PyX

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.9/628.9 kB 24.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 64.0 MB/s eta 0:00:00
  Created wheel for PyX: filename=pyx-0.17-py3-none-any.whl size=447521 sha256=e8f2cd61fbda0a6c7f847eacedeb2dbe3cbabaae343446b4dc6f6885e683fa12
  Stored in directory: /root/.cache/pip/wheels/c8/63/59/2e330d3e2eca0bdcf949ae145438bcc0ea3b9690dc4bd7107e
Successfully built PyX


In [4]:
from scapy.all import *

/usr/local/lib/python3.13/dist-packages/scapy/layers/tls/crypto/groups.py:25: CryptographyDeprecationWarning: Diffie-Hellman over finite fields (FFDH) is deprecated and support will be removed in a future release. Use a more modern key exchange algorithm.
  from cryptography.hazmat.primitives.asymmetric.dh import DHParameterNumbers


# Основы работы с библиотекой Scapy

# Манипуляции с пакетами

- пакеты это объекты
- оператор `/` используется для инкапсуляции пакетов

---

In [5]:
packet = IP() / TCP()
Ether() / packet

<Ether  type=IPv4 |<IP  frag=0 proto=tcp |<TCP  |>>>

In [ ]:
packet

<IP  frag=0 proto=6 |<TCP  |>>

In [6]:
#getlayer позволяет вывести конкретный вложенный протокол
packet.getlayer(TCP)

<TCP  |>

- функция `ls()` показывает все поля пакета (протокола)



In [7]:
ls(Ether)

dst        : DestMACField                        = ('None')
src        : SourceMACField                      = ('None')
type       : XShortEnumField                     = ('36864')


In [8]:
ls(IP)

version    : BitField  (4 bits)                  = ('4')
ihl        : BitField  (4 bits)                  = ('None')
tos        : XByteField                          = ('0')
len        : ShortField                          = ('None')
id         : ShortField                          = ('1')
flags      : FlagsField                          = ('<Flag 0 ()>')
frag       : BitField  (13 bits)                 = ('0')
ttl        : ByteField                           = ('64')
proto      : ByteEnumField                       = ('0')
chksum     : XShortField                         = ('None')
src        : SourceIPField                       = ('None')
dst        : DestIPField                         = ('None')
options    : PacketListField                     = ('[]')


In [9]:
ls(TCP)

sport      : ShortEnumField                      = ('20')
dport      : ShortEnumField                      = ('80')
seq        : IntField                            = ('0')
ack        : IntField                            = ('0')
dataofs    : BitField  (4 bits)                  = ('None')
reserved   : BitField  (3 bits)                  = ('0')
flags      : FlagsField                          = ('<Flag 2 (S)>')
window     : ShortField                          = ('8192')
chksum     : XShortField                         = ('None')
urgptr     : ShortField                          = ('0')
options    : TCPOptionsField                     = ("b''")


- вывести иерархию протоколов для пакета

In [11]:
packet.layers()

[scapy.layers.inet.IP, scapy.layers.inet.TCP]

In [10]:
[l.__name__ for l in packet.layers()]

['IP', 'TCP']

- В scapy реализовано автоматическое разрешение DNS имен и MAC адресов

---

In [12]:
p = Ether() / IP(dst="www.secdev.org") / TCP(flags="F")
p.summary()

'Ether / IP / TCP 172.28.0.12:ftp_data > Net("www.secdev.org/32"):http F'

In [ ]:
p.show()

###[ Ethernet ]###
  dst       = None
  src       = 02:42:ac:1c:00:0c
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = None
     tos       = 0x0
     len       = None
     id        = 1
     flags     = 
     frag      = 0
     ttl       = 64
     proto     = 6
     chksum    = None
     src       = 172.28.0.12
     dst       = Net("www.secdev.org/32")
     \options   \
###[ TCP ]###
        sport     = 20
        dport     = 80
        seq       = 0
        ack       = 0
        dataofs   = None
        reserved  = 0
        flags     = F
        window    = 8192
        chksum    = None
        urgptr    = 0
        options   = []



In [ ]:
p.show2()

###[ Ethernet ]###
  dst       = 02:42:b5:04:b6:7b
  src       = 02:42:ac:1c:00:0c
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 40
     id        = 1
     flags     = 
     frag      = 0
     ttl       = 64
     proto     = 6
     chksum    = 0x4388
     src       = 172.28.0.12
     dst       = 217.25.178.5
     \options   \
###[ TCP ]###
        sport     = 20
        dport     = 80
        seq       = 0
        ack       = 0
        dataofs   = 5
        reserved  = 0
        flags     = F
        window    = 8192
        chksum    = 0x5838
        urgptr    = 0
        options   = []



- доступ к различным полям пакета

---

In [17]:
print(p.src)      # первый слой, у которого есть src поле - Ethernet
print(p[IP].src)  # явное указание, что поле src читаем со слоя IP

# для sprintf() реализовано форматирование специфичных полей scapy
print(p.sprintf("%Ether.src% > %Ether.dst%\n%IP.src% > %IP.dst%"))

02:42:ac:1c:00:0c
172.28.0.12
02:42:ac:1c:00:0c > None
172.28.0.12 > Net("www.secdev.org/32")


принудительное вычисление значений всех полей пакета

In [18]:
raw_bytes = bytes(p)
computed = Ether(raw_bytes)
computed.dst

'02:42:ab:23:15:dd'

- генерация массивов пакетов с разными значениями полей

---

In [19]:
p_list = [p for p in IP(src="10.0.0.1", ttl=(1,5)) / ICMP()]  # диапазон для поля TTL
p_list

[<IP  frag=0 ttl=1 proto=icmp src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=2 proto=icmp src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=3 proto=icmp src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=4 proto=icmp src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=5 proto=icmp src=10.0.0.1 |<ICMP  |>>]

In [21]:
[p for p in IP() / TCP(dport=[22, 80, 443])]  # конкретный список значений для dport

[<IP  frag=0 proto=tcp |<TCP  dport=ssh |>>,
 <IP  frag=0 proto=tcp |<TCP  dport=http |>>,
 <IP  frag=0 proto=tcp |<TCP  dport=https |>>]

- проверка наличия заданного протокола в пакете

In [22]:
for p in p_list:
  if ICMP in p:
    print(p.ttl)

1
2
3
4
5


- преобразование массива пакетов в таблицу

In [23]:
import pandas as pd
packet_df = pd.DataFrame([p.fields for p in p_list if IP in p])
packet_df

,src,ttl,options
0,10.0.0.1,1,[]
1,10.0.0.1,2,[]
2,10.0.0.1,3,[]
3,10.0.0.1,4,[]
4,10.0.0.1,5,[]


# Живое взаимодействие с сетью

- функция `sr1()` позволяет отправить пакет и получить на него ответ
- Scapy автоматически сопоставляет запрос и ответ (если речь об известном протоколе)
    
---

In [24]:
p = sr1(IP(dst="8.8.8.8") / UDP() / DNS(rd=1, qd=DNSQR(qname="ya.ru", qtype="A")))
p[DNS].an


Received 2 packets, got 1 answers, remaining 0 packets


[<DNSRR  rrname=b'ya.ru.' type=A cacheflush=0 rclass=IN ttl=538 rdata=5.255.255.242 |>,
 <DNSRR  rrname=b'ya.ru.' type=A cacheflush=0 rclass=IN ttl=538 rdata=77.88.44.242 |>,
 <DNSRR  rrname=b'ya.ru.' type=A cacheflush=0 rclass=IN ttl=538 rdata=77.88.55.242 |>]

- Функция `sniff()` позволяет осуществлять захват трафика

---

In [25]:
help(sniff)

Help on function sniff in module scapy.sendrecv:

sniff(*args, **kwargs)
    Sniff packets and return a list of packets.

    Args:
        count: number of packets to capture. 0 means infinity.
        store: whether to store sniffed packets or discard them
        prn: function to apply to each packet. If something is returned, it
             is displayed.
             --Ex: prn = lambda x: x.summary()
        session: a session = a flow decoder used to handle stream of packets.
                 --Ex: session=TCPSession
                 See below for more details.
        filter: BPF filter to apply.
        lfilter: Python function applied to each packet to determine if
                 further action may be done.
                 --Ex: lfilter = lambda x: x.haslayer(Padding)
        offline: PCAP file (or list of PCAP files) to read packets from,
                 instead of sniffing them
        quiet:   when set to True, the process stderr is discarded
                 (default: 

In [26]:
s = sniff(count=2)
s

<Sniffed: TCP:2 UDP:0 ICMP:0 Other:0>

In [27]:
sniff(count=2, prn=lambda p: p.summary())

Ether / IP / TCP 172.28.0.1:53978 > 172.28.0.12:x11 PA / Raw
Ether / IP / TCP 172.28.0.12:x11 > 172.28.0.1:53978 PA / Raw


<Sniffed: TCP:2 UDP:0 ICMP:0 Other:0>

- пакеты можно прочитать из pcap файла

---

In [28]:
pcap_p = rdpcap("/content/drive/MyDrive/DataAnalysis/http_espn_fail.pcapng")
len(pcap_p)

569

- метод `command()` выводит строку, которая позволяет создать точно такой же пакет

---

In [29]:
pcap_p[0].command()

"Ether(dst='c0:c1:c0:17:8c:e8', src='78:31:c1:cb:b2:56', type=2048)/IP(version=4, ihl=5, tos=0, len=58, id=9480, flags=0, frag=0, ttl=64, proto=17, chksum=37630, src='172.16.16.154', dst='4.2.2.1')/UDP(sport=57434, dport=53, len=38, chksum=31007)/DNS(id=45323, qr=0, opcode=0, aa=0, tc=0, rd=1, ra=0, z=0, ad=0, cd=0, rcode=0, qdcount=1, ancount=0, nscount=0, arcount=0, qd=[DNSQR(qname=b'www.espn.com.', qtype=1, unicastresponse=0, qclass=1)])"

- анализ протоколов в pcap файле

In [30]:
def get_proto_list(pcap_p):
  proto_list = set()
  for packet in pcap_p:
    proto_list = proto_list | set(l.__name__ for l  in packet.layers())
  return proto_list

In [31]:
get_proto_list(pcap_p)

{'DNS', 'Ether', 'IP', 'Padding', 'Raw', 'TCP', 'UDP'}

- использоване `sniff()` в offline режиме позволяет фильтровать уже имеющийся pcap файл

In [34]:
pcap_p = sniff(offline="/content/drive/MyDrive/DataAnalysis/http_espn_fail.pcapng", filter="port 53")
len(pcap_p)

14

In [33]:
!apt update
!apt install tcpdump

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,019 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,905 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Pack

In [35]:
get_proto_list(pcap_p)

{'DNS', 'Ether', 'IP', 'UDP'}

In [36]:
pcap_p[1][DNS].an

[<DNSRR  rrname=b'www.espn.com.' type=CNAME cacheflush=0 rclass=IN ttl=225 rdata=b'redir.espn.gns.go.com.' |>,
 <DNSRR  rrname=b'redir.espn.gns.go.com.' type=A cacheflush=0 rclass=IN ttl=223 rdata=68.71.212.158 |>]

- Функция `lsc()` выводит список доступных команд (функций)

---

```
>>> lsc()
IPID_count          : Identify IP id values classes in a list of packets
arpcachepoison      : Poison target's cache with (your MAC,victim's IP) couple
arping              : Send ARP who-has requests to determine which hosts are up
bind_layers         : Bind 2 layers on some specific fields' values
bridge_and_sniff    : Forward traffic between interfaces if1 and if2, sniff and return the
chexdump            :  Build a per byte hexadecimal representation
computeNIGroupAddr  : Compute the NI group Address. Can take a FQDN as input parameter
corrupt_bits        : Flip a given percentage or number of bits from a string
corrupt_bytes       : Corrupt a given percentage or number of bytes from a string
defrag              : defrag(plist) -> ([not fragmented], [defragmented],
defragment          : defrag(plist) -> plist defragmented as much as possible
dhcp_request        : --
[..]
```

In [37]:
lsc()

IPID_count            : Identify IP id values classes in a list of packets
arp_mitm              : ARP MitM: poison 2 target's ARP cache
arpcachepoison        : Poison targets' ARP cache
arping                : Send ARP who-has requests to determine which hosts are up::
arpleak               : Exploit ARP leak flaws, like NetBSD-SA2017-002.
bind_layers           : Bind 2 layers on some specific fields' values.
bridge_and_sniff      : Forward traffic between interfaces if1 and if2, sniff and return
chexdump              : Build a per byte hexadecimal representation
computeNIGroupAddr    : Compute the NI group Address. Can take a FQDN as input parameter
connect_from_ip       : Open a TCP socket to a host:port while spoofing another IP.
corrupt_bits          : Flip a given percentage (at least one bit) or number of bits
corrupt_bytes         : Corrupt a given percentage (at least one byte) or number of bytes
dclocator             : Perform a DC Locator as per [MS-ADTS] sect 6.3.6 or RFC41

In [38]:
from collections import defaultdict

# Итоговый словарь: {нормализованный 5-tuple: {"forward": N, "backward": N}}
biflows = defaultdict(lambda: {"forward": 0, "backward": 0})

for p in pcap_p:
    if IP not in p:
        continue

    proto = p[IP].proto

    if TCP in p:
        sport, dport = p[TCP].sport, p[TCP].dport
    elif UDP in p:
        sport, dport = p[UDP].sport, p[UDP].dport
    else:
        sport, dport = 0, 0

    src, dst = p[IP].src, p[IP].dst

    # Нормализация ключа — меньший IP всегда первый
    if (src, sport) < (dst, dport):
        key = (src, dst, proto, sport, dport)
        biflows[key]["forward"] += len(p[IP])
    else:
        key = (dst, src, proto, dport, sport)
        biflows[key]["backward"] += len(p[IP])

In [41]:
biflows

defaultdict(<function __main__.<lambda>()>,
            {('172.16.16.154', '4.2.2.1', 17, 57434, 53): {'forward': 58,
              'backward': 106},
             ('172.16.16.154', '4.2.2.1', 17, 22689, 53): {'forward': 57,
              'backward': 96},
             ('172.16.16.154', '4.2.2.1', 17, 52723, 53): {'forward': 64,
              'backward': 146},
             ('172.16.16.154', '4.2.2.1', 17, 23201, 53): {'forward': 60,
              'backward': 163},
             ('172.16.16.154', '4.2.2.1', 17, 57920, 53): {'forward': 64,
              'backward': 171},
             ('172.16.16.154', '4.2.2.1', 17, 49283, 53): {'forward': 60,
              'backward': 163},
             ('172.16.16.154', '4.2.2.1', 17, 21916, 53): {'forward': 60,
              'backward': 163}})


# Визуализация

- функция `bytes()` позволяет перевести пакет в его бинарное представление (как будет передан в сеть)

In [42]:
pkt = IP() / UDP() / DNS(qd=DNSQR())
repr(bytes(pkt))

"b'E\\x00\\x00=\\x00\\x01\\x00\\x00@\\x11|\\xad\\x7f\\x00\\x00\\x01\\x7f\\x00\\x00\\x01\\x005\\x005\\x00)\\xb6\\xd3\\x00\\x00\\x01\\x00\\x00\\x01\\x00\\x00\\x00\\x00\\x00\\x00\\x03www\\x07example\\x03com\\x00\\x00\\x01\\x00\\x01'"

Также scapy может вывести пакет в виде шестнадцатиричного дампа
   -  `hexdump` позволяет вывести содрежимое в human-readable варианте

In [43]:
hexdump(pkt)

0000  45 00 00 3D 00 01 00 00 40 11 7C AD 7F 00 00 01  E..=....@.|.....
0010  7F 00 00 01 00 35 00 35 00 29 B6 D3 00 00 01 00  .....5.5.)......
0020  00 01 00 00 00 00 00 00 03 77 77 77 07 65 78 61  .........www.exa
0030  6D 70 6C 65 03 63 6F 6D 00 00 01 00 01           mple.com.....


In [44]:
pkt.show()

###[ IP ]###
  version   = 4
  ihl       = None
  tos       = 0x0
  len       = None
  id        = 1
  flags     = 
  frag      = 0
  ttl       = 64
  proto     = udp
  chksum    = None
  src       = 127.0.0.1
  dst       = 127.0.0.1
  \options   \
###[ UDP ]###
     sport     = domain
     dport     = domain
     len       = None
     chksum    = None
###[ DNS ]###
        id        = 0
        qr        = 0
        opcode    = QUERY
        aa        = 0
        tc        = 0
        rd        = 1
        ra        = 0
        z         = 0
        ad        = 0
        cd        = 0
        rcode     = ok
        qdcount   = None
        ancount   = None
        nscount   = None
        arcount   = None
        \qd        \
         |###[ DNS Question Record ]###
         |  qname     = b'www.example.com.'
         |  qtype     = A
         |  unicastresponse= 0
         |  qclass    = IN
        \an        \
        \ns        \
        \ar        \



## Простой ответчик

Scapy can wait for a query, then send an answer with the `AnsweringMachine` object.

Two methods are mandatory:
1. `is_request()`: returns `True` if the packet is the expected query
2. `make_reply()`: returns the packet that will be sent by Scapy


In [61]:
# Create the Answering Machine
class TestResponse(AnsweringMachine):
  function_name = "test"
  dst_addr = ""
  def is_request(self, pkt):
    if IP in pkt:
      dst_addr = pkt[IP].dst
    return ICMP in pkt

  def make_reply(self, req):

    rep = IP()/ICMP(type=0,code=0)
    return rep

# Start the answering machine
# TestResponse()()

# Работа с сертификатами X.509

- анализ и вывод содержимого сертификата X.509
---

In [62]:
load_layer("tls")
cert_github = Cert(open("/content/drive/MyDrive/DataAnalysis/github.pem").read())  # assuming you d/l the certificate
cert_github

[X.509 Cert. Subject:/CN=github.com, Issuer:/C=GB/O=Sectigo Limited/CN=Sectigo Public Server Authentication CA DV E36]

- Некоторые методы работы с сертификатом

---

In [63]:
print(cert_github.isSelfSigned())  # check if it is self signed
print(cert_github.subject)  # display the subject
print(cert_github.remainingDays()) # compute the number of days until expiration

False
{'commonName': 'github.com'}
-113.67483796296297


# Работа с трафиком TLS

In [64]:
load_layer("tls")
s = rdpcap("/content/drive/MyDrive/DataAnalysis/tls.pcapng")
ch_list = [p for p in s if TLSClientHello in p]  # найти сообщения Client Hello
len(ch_list)

1

In [65]:
for p in ch_list:
  p[TLSClientHello].show()  # display the first message

###[ TLS Handshake - Client Hello ]###
  msgtype   = client_hello
  msglen    = 774
  version   = TLS 1.2
  gmt_unix_time= Sun, 28 May 2102 20:12:20  (4178290340)
  random_bytes= db608a0658107674bc1b5515c6d65db4701f4db59e9ac8a4b0017930
  sidlen    = 32
  sid       = b')\xf0\x0e\x95\x0e \x86\xac>\xca\x97\xbdU\x8dn\x05Mt\xe0\x0c\x9b,;Ie\x19\xb3\x12\x1f\xa8\xe3\xf6'
  cipherslen= 40
  ciphers   = [TLS_AES_256_GCM_SHA384, TLS_AES_128_GCM_SHA256, TLS_ECDHE_ECDSA_WITH_AES_256_GCM_SHA384, TLS_ECDHE_ECDSA_WITH_AES_128_GCM_SHA256, TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384, TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256, TLS_ECDHE_ECDSA_WITH_AES_256_CBC_SHA384, TLS_ECDHE_ECDSA_WITH_AES_128_CBC_SHA256, TLS_ECDHE_RSA_WITH_AES_256_CBC_SHA384, TLS_ECDHE_RSA_WITH_AES_128_CBC_SHA256, TLS_ECDHE_ECDSA_WITH_AES_256_CBC_SHA, TLS_ECDHE_ECDSA_WITH_AES_128_CBC_SHA, TLS_ECDHE_RSA_WITH_AES_256_CBC_SHA, TLS_ECDHE_RSA_WITH_AES_128_CBC_SHA, TLS_RSA_WITH_AES_256_GCM_SHA384, TLS_RSA_WITH_AES_128_GCM_SHA256, TLS_RSA_WITH_AES_256